# Training and Fine-Tuning BERT for Classification
## Classfying Goodreads Reviews By Book Genre

By Maria Antoniak, Melanie Walsh, and the [AI for Humanists](https://aiforhumanists.com/) Team

Updated: 2024-11-05
<br></br>

This notebook will demonstrate how users can train and fine-tune a BERT model for classification with the popular HuggingFace `transformers` Python library.

We will fine-tune a BERT model on Goodreads reviews from the [UCSD Book Graph](https://mengtingwan.github.io/data/goodreads.html) with the goal of predicting the genre of the book being reviewed. The genres include:
- poetry
- comics & graphic
- fantasy & paranormal
- history & biography
- mystery, thriller, & crime
- romance
- young adult  

**Basic steps involved in using BERT and HuggingFace:**
1. Divide your data into training and test sets.
2. Encode your data into a format BERT will understand.
3. Combine your data and labels into datset objects.
4. Load the pre-trained BERT model.
5. Fine-tune the model using your training data.
6. Predict new labels and evaluate performance on your test data.



<br><br>

## **Import necessary Python libraries and modules**

First, we will import necessary Python libraries and modules. These include as `gdown`, for downloading large files from Google Drive (where we will get our UCSD Goodreads reviews), as well as scikit-learn (`sklearn`) and PyTorch (`torch`), for various machine learning tools.

In [ ]:
!pip3 install -U transformers evaluate datasets accelerate

In [ ]:
# Basic Python modules
from collections import defaultdict
import random
# import pickle

# For downloading large files from Google Drive
# https://github.com/wkentaro/gdown
import gdown

# For working with gzip files
# https://docs.python.org/3/library/gzip.html
import gzip

# For working with JSON files
import json

# For data manipulation and analysis
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# For deep learning
# https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html
import torch

# For plotting and data visualization
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker
sns.set(style='ticks', font_scale=1.2)

The HuggingFace [`transformers` Python library](https://huggingface.co/transformers/installation.html) is included in Colab by default now, so we do not need to install it (but this is how you would install it with `pip`).

From `transformers`, we will import modules for `DistilBert`, a *distilled* or smaller version of a BERT model that runs more quickly and uses less computing power. This makes it ideal for those just getting started with BERT.

In [ ]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

<br><br>

## **Set parameters and file paths**

In [ ]:
# # This is the name of the BERT model that we want to use.
# # We're using DistilBERT to save space (it's a distilled version of the full BERT model),
# # and we're going to use the cased (vs uncased) version.
# model_name = 'distilbert-base-cased'

# # This is the name of the program management system for NVIDIA GPUs. We're going to send our code here.
# device_name = 'cuda'

# # This is the maximum number of tokens in any document sent to BERT.
# max_length = 512

# # This is the name of the directory where we'll save our model. You can name it whatever you want.
# cached_model_directory_name = 'distilbert-reviews-genres'

<br><br>

## **Load and sample Goodreads data**

In this cell, we create a Python dictionary with each genre and the link to the corresponding UCSD Goodreads review data for that genre.

*If you manually click on any of the URLs, you will be able to download the data for that genre. For example, here's the link for poetry: https://datarepo.eng.ucsd.edu/mcauley_group/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz*

In [ ]:
# This is where our target data is hosted on the web. You only need these paths for the book review dataset.

# Source: https://mengtingwan.github.io/data/goodreads.html#datasets

genre_url_dict = {'poetry':                 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz',
                  'children':               'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz',
                  'comics_graphic':         'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz',
                  'fantasy_paranormal':     'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz',
                  'history_biography':      'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz',
                  'mystery_thriller_crime': 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz',
                  'romance':                'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz',
                  'young_adult':            'https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz'}

Next we loop through this dictionary and use `gdown` to download the Goodreads review data for each genre from Google Drive.

Now we will load the first 100,000 reviews from each link and randomly sample 2,000 reviews.

In [ ]:
import requests
# Stream reviews from URL and collect a subset
def load_reviews(url, head=10000, sample_size=2000):
    reviews = []
    count = 0

    response = requests.get(url, stream=True)
    print(response)
    with gzip.open(response.raw, 'rt', encoding='utf-8') as file:
        for line in file:
            d = json.loads(line)
            reviews.append(d['review_text'])
            count += 1

            # Stop if we have reached the 100,000 limit
            if head is not None and count >= head:
                break

    # Return random sample of reviews
    return random.sample(reviews, min(sample_size, len(reviews)))

# Reviews by genre
genre_reviews_dict = {}

# Load reviews for each genre
for genre, url in genre_url_dict.items():
    print(f'Loading reviews for genre: {genre}')
    genre_reviews_dict[genre] = load_reviews(url, head=10000, sample_size=2000)


Let's preview a couple of the key-value pairs in `genre_reviews_dict`

In [ ]:
 for _genre, _reviews in genre_reviews_dict.items():
    print(f"\n\n{_genre}\n")
    print(random.sample(_reviews, 1)[0])

Here we use `pickle` to save this Python dictionary to a `.pickle` file so we can easily load it later.

*The `pickle` module allows you to save and load Python objects like lists and dictionaries.*

In [ ]:
# pickle.dump(genre_reviews_dict, open('genre_reviews_dict.pickle', 'wb'))
# genre_reviews_dict = pickle.load(open('genre_reviews_dict.pickle', 'rb'))
import pickle

# To save (Write Binary)
with open('genre_reviews_dict.pickle', 'wb') as f:
    pickle.dump(genre_reviews_dict, f)

# To load (Read Binary)
# with open('genre_reviews_dict.pickle', 'rb') as f:
#     genre_reviews_dict = pickle.load(f)

> Task 2. Load a pre-trained model from hugging face

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Define the pre-trained model checkpoint from Hugging Face Hub
model_checkpoint = "distilbert-base-uncased"

# 1. Load the tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 2. Define the number of output labels for your specific task
# Change this number to match dataset's unique categories or ratings)
NUM_LABELS = len(genre_reviews_dict)

# 3. Load the model with the correct number of output labels [3 Marks]
print(f"Loading model with {NUM_LABELS} output labels...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=NUM_LABELS
)

print("\nModel and Tokenizer successfully loaded!")

> Task 2. Train the model and track with W&B

In [ ]:
import pandas as pd
from datasets import Dataset

# 1. Flatten your dictionary into a list of rows
all_rows = []
for genre, reviews in genre_reviews_dict.items():
    for review in reviews:
        # Adjust these keys ('reviewText' and 'overall') to match actual data columns
        all_rows.append({
            "text": str(review),
            "label_raw": genre  # Using rating as an example
        })

# Convert to a Pandas DataFrame
df_all = pd.DataFrame(all_rows)

# 2. Convert raw labels to numerical indices (e.g., 1-5 stars to 0-4 indices)
# If your task is binary sentiment (e.g., Positive=1, Negative=0), adapt this step!
df_all['label'] = df_all['label_raw'].astype('category').cat.codes
NUM_LABELS = df_all['label'].nunique()

# Convert the Pandas DataFrame into a Hugging Face Dataset
raw_dataset = Dataset.from_pandas(df_all)

# 3. Split into Train (80%) and Validation (20%) datasets
split_dataset = raw_dataset.train_test_split(test_size=0.2, seed=42)

# 4. Tokenize the text data
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing the datasets...")
tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

# Remove unneeded raw columns and ensure format is ready for PyTorch
tokenized_datasets = tokenized_datasets.remove_columns(["text", "label_raw"])
tokenized_datasets.set_format("torch")

train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"]

print(f"Data ready! Train size: {len(train_dataset)}, Validation size: {len(eval_dataset)}")

In [ ]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

# 1. Define compute_metrics for Accuracy and F1 [4 Marks]
def compute_metrics(eval_pred):
    metric_acc = evaluate.load("accuracy")
    metric_f1 = evaluate.load("f1")
    
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    # Use 'macro' or 'binary' depending on your number of labels
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    
    return {"accuracy": acc, "f1": f1}

# 2. Configure Training Arguments [4 Marks]
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,              # Train for 3 epochs
    per_device_train_batch_size=16,   # Batch size for training
    per_device_eval_batch_size=16,    # Batch size for evaluation
    eval_strategy="epoch",           # Evaluate at the end of each epoch
    save_strategy="epoch",           # Save checkpoints at the end of each epoch
    logging_steps=50,                # Send logs to W&B every 50 steps
    learning_rate=5e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="wandb",               # << CRITICAL: Connects directly to W&B
    run_name="distilbert-genre-analysis" # Name of your run in W&B dashboard
)

In [ ]:
# Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# Initialize the W&B run using your environment token from Task 1
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Safe token ingestion from Kaggle Secrets
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb_api_key")

# Start training
print("Starting training loop...")
trainer.train()

# Properly finish the W&B logging session
wandb.finish()
print("Training complete and metrics synced successfully!")

> Task 3: Evaluate and save results

In [ ]:
import json
import wandb
from sklearn.metrics import classification_report

# 1. Re-initialize W&B because we closed it in the last cell
# (Change the project name if you used a specific one earlier!)
wandb.init(project="mlops-assignment2", name="final-evaluation", job_type="eval")

# 2. Run evaluation on the test/eval set
print("Running final evaluation...")
eval_results = trainer.evaluate()
print("\nFinal Evaluation Metrics:\n", eval_results)

# 3. Log final metrics to W&B explicitly 
wandb.log({
    "final/loss":     eval_results.get("eval_loss"),
    "final/accuracy": eval_results.get("eval_accuracy"),
    "final/f1":       eval_results.get("eval_f1"),
})

# 4. Generate predictions to build the classification report
prediction_output = trainer.predict(eval_dataset)
preds = prediction_output.predictions.argmax(-1)
true_labels = prediction_output.label_ids

# Extract the original category names for the report
target_names = df_all['label_raw'].astype('category').cat.categories.astype(str).tolist()

# 5. Save the classification report as a file
report = classification_report(
    true_labels, 
    preds, 
    target_names=target_names, 
    output_dict=True
)

with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("\nClassification report saved to 'eval_report.json'")

# 6. Upload to W&B as a versioned Artifact 
print("Uploading evaluation report artifact to Weights & Biases...")
artifact = wandb.Artifact("eval-report", type="evaluation")
artifact.add_file("eval_report.json")
wandb.log_artifact(artifact)

# Close the W&B run properly
wandb.finish()
print("\nTask Complete! Run successfully finished and logged.")

> Task 4: Save model to hugging face hub

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import wandb

# STEP 1: Authenticate with Hugging Face
# We use Kaggle Secrets so our private HF token isn't visible if we share this notebook.
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token") 

# Log in to the Hugging Face Hub so we have permission to push files to our account
login(token=hf_token)

# STEP 2: Define your repository
# 'rahul-solanki' is actual Hugging Face username!
repo_name = "rahul-solanki/distilbert-goodreads-genres"

print(f"Pushing model and tokenizer to Hugging Face Hub: {repo_name}...")

# STEP 3: Push Model and Tokenizer
# push_to_hub automatically creates the repo if it doesn't exist.
# We push the model (the trained weights) so others can use its predictions.
model.push_to_hub(repo_name)

# We push the tokenizer so others can convert their raw text into the exact 
# numerical format this specific model expects.
tokenizer.push_to_hub(repo_name)

# STEP 4: Log the URL to Weights & Biases
# We construct the public URL where our model now lives.
hf_model_url = f"https://huggingface.co/{repo_name}"

# We save this URL into the 'summary' of our current W&B run. 
# This links our training experiment directly to our final deployed model!
if wandb.run is not None:
    wandb.run.summary["huggingface_model"] = hf_model_url
    print(f"Successfully logged Hugging Face URL to W&B: {hf_model_url}")
else:
    print("W&B run is not active, but model pushed successfully!")

# Now it is safe to close the W&B run.
wandb.finish()

> Task 5: Push Everything to GitHub & READMD

**Requirement.txt**
### pandas
### numpy
### torch
### transformers
### datasets
### evaluate
### accelerate
### wandb
### huggingface_hub
### scikit-learn
### matplotlib
### seaborn
### requests
### gdown

**Readme.md**
# Goodreads Genre Classifier (MLOps Pipeline)

This repository contains an end-to-end MLOps pipeline that fine-tunes a `distilbert-base-uncased` transformer model to classify Goodreads book reviews into their respective genres. The training was orchestrated on Kaggle using GPU acceleration, tracked via Weights & Biases (W&B), and the final model is hosted on the Hugging Face Hub.

## Setup Instructions
To run this project locally or reproduce the environment:
1. Clone this repository: `git clone https://github.com/your-username/your-repo-name.git`
2. Install the dependencies: `pip install -r requirements.txt`
3. Set your environment variables for `WANDB_API_KEY` and `HF_TOKEN`.
4. Run the provided Jupyter Notebook.

## Results
The model was evaluated on a 20% test split. Here are the final metrics:

| Metric    | Score |
|-----------|-------|
| Accuracy  | 0.608 | 
| F1 Score  | 0.609 | 
| Eval Loss | 2.207 | 


## Project Links
- **Training Platform (Kaggle Notebook):** https://www.kaggle.com/code/rahulsolankijodhpur/mlops-assignment2
- **Experiment Tracking (W&B Dashboard):** https://wandb.ai/rahul_solanki-prom-iit-rajasthan/mlops-assignment2/reports/Untitled-Report--VmlldzoxNzAzMjMxNg?accessToken=9mupvcl2m1pv1880ermqah87zpq3p55almasuc3eii1bca2msfmla4p463x2gbz8
  https://wandb.ai/rahul_solanki-prom-iit-rajasthan/huggingface?nw=nwuserrahul_solanki & https://wandb.ai/rahul_solanki-prom-iit-rajasthan/mlops-assignment2
- **Deployed Model (Hugging Face):** https://huggingface.co/rahul-solanki/distilbert-goodreads-genres


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Generate the confusion matrix data
cm = confusion_matrix(true_labels, preds)

# 2. Set up the plot aesthetics
plt.figure(figsize=(10, 8))
sns.set(font_scale=1.1) # Make the font a bit larger for readability

# 3. Create the heatmap using Seaborn
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, 
            yticklabels=target_names,
            linewidths=0.5, linecolor='gray')

# 4. Add titles and labels
plt.title('Confusion Matrix: Goodreads Genre Classification', fontsize=16, pad=20)
plt.xlabel('Predicted Genre', fontsize=14, labelpad=10)
plt.ylabel('True / Actual Genre', fontsize=14, labelpad=10)

# Rotate x-axis labels so they don't overlap
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

# 5. Display the plot
plt.tight_layout()
plt.show()